In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, MaxNLocator
import seaborn as sns
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.style.use('default')
sns.set_palette("husl")

plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

In [4]:
# Load the data

data_path = Path('../data/processed/clean/combined_individual_events.csv')
df = pd.read_csv(data_path)

print(f"Dataset Shape: {df.shape}")
print(f"Columns: {df.columns}")
print('--' * 40)

df.head()


Dataset Shape: (345, 17)
Columns: Index(['year', 'event_name', 'stroke', 'gender', 'distance', 'meet',
       'source_file', 'winning_time_sec', 'winning_time_format',
       'a_final_cutoff_sec', 'a_final_cutoff_format', 'b_final_cutoff_sec',
       'b_final_cutoff_format', 'c_final_cutoff_sec', 'c_final_cutoff_format',
       'total_swimmers', 'results'],
      dtype='object')
--------------------------------------------------------------------------------


,year,event_name,stroke,gender,distance,meet,source_file,winning_time_sec,winning_time_format,a_final_cutoff_sec,a_final_cutoff_format,b_final_cutoff_sec,b_final_cutoff_format,c_final_cutoff_sec,c_final_cutoff_format,total_swimmers,results
0,2002,100_Backstroke,Backstroke,Men,100,2002_Nescac_Msd_Results,2002_NESCAC_MSD_Results.txt,51.45,51.45,54.94,54.94,56.16,56.16,NaN,NaN,23,"[{'name': 'Schwenker, Eric', 'yr': 'SR', 'scho..."
1,2002,100_Breaststroke,Breaststroke,Men,100,2002_Nescac_Msd_Results,2002_NESCAC_MSD_Results.txt,56.98,56.98,59.83,59.83,61.58,1:01.58,63.14,1:03.14,35,"[{'name': 'Eck, Jonathan', 'yr': 'JR', 'school..."
2,2002,100_Butterfly,Butterfly,Men,100,2002_Nescac_Msd_Results,2002_NESCAC_MSD_Results.txt,50.70,50.70,53.04,53.04,54.46,54.46,56.05,56.05,28,"[{'name': 'Stuntz, Grayson', 'yr': 'SR', 'scho..."
3,2002,100_Freestyle,Freestyle,Men,100,2002_Nescac_Msd_Results,2002_NESCAC_MSD_Results.txt,45.83,45.83,48.14,48.14,48.79,48.79,49.54,49.54,45,"[{'name': 'Walendziak, Nick', 'yr': 'SO', 'sch..."
4,2002,200_Backstroke,Backstroke,Men,200,2002_Nescac_Msd_Results,2002_NESCAC_MSD_Results.txt,111.13,1:51.13,118.61,1:58.61,121.44,2:01.44,135.04,2:15.04,24,"[{'name': 'Schwenker, Eric', 'yr': 'SR', 'scho..."


In [5]:
print(f"Unique events: {df['event_name'].value_counts()}")

print('--' * 40)

print(f"Year Range: {df['year'].min()} - {df['year'].max()}")

Unique events: event_name
100_Backstroke      23
100_Breaststroke    23
100_Butterfly       23
100_Freestyle       23
200_Backstroke      23
200_Breaststroke    23
200_Butterfly       23
200_Freestyle       23
200_IM              23
400_IM              23
500_Freestyle       23
50_Backstroke       23
50_Breaststroke     23
50_Butterfly        23
50_Freestyle        23
Name: count, dtype: int64
--------------------------------------------------------------------------------
Year Range: 2002 - 2025


In [6]:
# Datatypes
print("Datatypes:")
print(f"A_cutoff_sec: {df['a_final_cutoff_sec'].dtype}")
print(f"B_cutoff_sec: {df['b_final_cutoff_sec'].dtype}")
print(f"C_cutoff_sec: {df['c_final_cutoff_sec'].dtype}")

# Convert to numeric just in case

for col in ['a_final_cutoff_sec', 'b_final_cutoff_sec', 'c_final_cutoff_sec']:
    df[col] = pd.to_numeric(df[col], errors = 'coerce')

print("After Conversion:")
print(f"A_cutoff_sec: {df['a_final_cutoff_sec'].dtype}")
print(f"B_cutoff_sec: {df['b_final_cutoff_sec'].dtype}")
print(f"C_cutoff_sec: {df['c_final_cutoff_sec'].dtype}")

Datatypes:
A_cutoff_sec: float64
B_cutoff_sec: float64
C_cutoff_sec: float64
After Conversion:
A_cutoff_sec: float64
B_cutoff_sec: float64
C_cutoff_sec: float64


In [7]:
# Unique ID

df['event_id'] = df['gender'] + '_' + df['event_name']

print(f"Unique events: {df['event_id'].value_counts()}")




Unique events: event_id
Men_100_Backstroke      23
Men_100_Breaststroke    23
Men_100_Butterfly       23
Men_100_Freestyle       23
Men_200_Backstroke      23
Men_200_Breaststroke    23
Men_200_Butterfly       23
Men_200_Freestyle       23
Men_200_IM              23
Men_400_IM              23
Men_500_Freestyle       23
Men_50_Backstroke       23
Men_50_Breaststroke     23
Men_50_Butterfly        23
Men_50_Freestyle        23
Name: count, dtype: int64


In [8]:
def sec_to_time(x, pos):
    minutes = int(x // 60)
    seconds = x % 60
    return f"{minutes}:{seconds:05.2f}"

def plot_event_cutoffs(event_data, event_name, output_dir):
    # sort by year
    event_data = event_data.sort_values('year')

    # compute year bounds
    min_year = int(event_data['year'].min()) - 1
    max_year = int(event_data['year'].max()) + 1

    # make a “pretty” title (no underscores)
    pretty_name = event_name.replace('_', ' ')

    fig, ax = plt.subplots(figsize=(12, 8))

    # A final cutoff: drop NaNs so the line connects across gaps
    if not event_data['a_final_cutoff_sec'].isna().all():
        a = event_data[['year', 'a_final_cutoff_sec']].dropna()
        ax.plot(
            a['year'],
            a['a_final_cutoff_sec'],
            marker='o',
            linestyle='-',
            linewidth=2,
            markersize=6,
            label='A Final Cutoff',
            color='#1f77b4'
        )

    # B final cutoff: drop NaNs
    if not event_data['b_final_cutoff_sec'].isna().all():
        b = event_data[['year', 'b_final_cutoff_sec']].dropna()
        ax.plot(
            b['year'],
            b['b_final_cutoff_sec'],
            marker='s',
            linestyle='--',
            linewidth=2,
            markersize=6,
            label='B Final Cutoff',
            color='#ff7f0e'
        )

    # C final cutoff: drop NaNs
    if not event_data['c_final_cutoff_sec'].isna().all():
        c = event_data[['year', 'c_final_cutoff_sec']].dropna()
        ax.plot(
            c['year'],
            c['c_final_cutoff_sec'],
            marker='^',
            linestyle='-.',
            linewidth=2,
            markersize=6,
            label='C Final Cutoff',
            color='#17becf'
        )

    # clean up spines & add a light grid
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)

    ax.yaxis.set_major_formatter(FuncFormatter(sec_to_time))

    # titles & labels
    ax.set_title(f'{pretty_name} Final Cutoffs by Year',
                 fontsize=18, fontweight='bold', pad=15)
    ax.set_xlabel('Year', fontsize=14)
    ax.set_ylabel('Time', fontsize=14)

    # x‑ticks every 4 years
    years = list(range(min_year, max_year + 1, 4))
    ax.set_xlim(min_year, max_year)
    ax.set_xticks(years)
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=12)
    plt.setp(ax.get_yticklabels(), fontsize=12)

    # legend without box
    ax.legend(frameon=False, fontsize=12, loc='best')

    # ensure nothing gets cut off
    plt.tight_layout()

    # save using underscores in filename
    safe_name = event_name.replace(' ', '_').replace('/', '_')
    output_path = Path(output_dir) / f'{safe_name}_cutoffs.png'
    fig.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

    return output_path

In [9]:
output_dir = Path('../output/plots/event_cutoffs')
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Creating plots in: {output_dir.absolute()}")

Creating plots in: /Users/hlecates/Desktop/aqua-analytics/nescac/notebooks/../output/plots/event_cutoffs


In [10]:
saved_plots = []

for event_id in df['event_id'].unique():
    event_data = df[df['event_id'] == event_id]

    if event_data[['a_final_cutoff_sec', 'b_final_cutoff_sec', 'c_final_cutoff_sec']].isna().all().all():
        continue

    plot_title = f'{event_id}'

    try:
        plot_path = plot_event_cutoffs(event_data, plot_title, output_dir)
        saved_plots.append(plot_path)
        print(f"Created plot: {plot_title}")
    except Exception as e:
        print(f"Error creating plot for {plot_title}: {e}")

print(f"Total plots created: {len(saved_plots)}")

Created plot: Men_100_Backstroke
Created plot: Men_100_Breaststroke
Created plot: Men_100_Butterfly
Created plot: Men_100_Freestyle
Created plot: Men_200_Backstroke
Created plot: Men_200_Breaststroke
Created plot: Men_200_Butterfly
Created plot: Men_200_Freestyle
Created plot: Men_200_IM
Created plot: Men_400_IM
Created plot: Men_500_Freestyle
Created plot: Men_50_Backstroke
Created plot: Men_50_Breaststroke
Created plot: Men_50_Butterfly
Created plot: Men_50_Freestyle
Total plots created: 15


In [ ]:
def plot_cutoffs_grid(event_data_dict, output_dir, nrows=5, ncols=3):
    all_years = pd.concat([df['year'] for df in event_data_dict.values()])
    min_year = int(all_years.min()) - 1
    max_year = int(all_years.max()) + 1
    years_tick = list(range(min_year, max_year + 1, 4))

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(ncols * 4, nrows * 3),
        sharex=True,
        sharey=False
    )
    axes_flat = axes.flatten()

    for ax, (event_name, df) in zip(axes_flat, event_data_dict.items()):
        df = df.sort_values('year')

        # A‑final
        if not df['a_final_cutoff_sec'].isna().all():
            a = df[['year','a_final_cutoff_sec']].dropna()
            ax.plot(
                a['year'], a['a_final_cutoff_sec'],
                marker='o', linestyle='-', linewidth=1, markersize=3,
                label='A Final', color='#1f77b4'
            )

        # B‑final
        if not df['b_final_cutoff_sec'].isna().all():
            b = df[['year','b_final_cutoff_sec']].dropna()
            ax.plot(
                b['year'], b['b_final_cutoff_sec'],
                marker='s', linestyle='--', linewidth=1, markersize=3,
                label='B Final', color='#ff7f0e'
            )

        # C‑final
        if not df['c_final_cutoff_sec'].isna().all():
            c = df[['year','c_final_cutoff_sec']].dropna()
            ax.plot(
                c['year'], c['c_final_cutoff_sec'],
                marker='^', linestyle='-.', linewidth=1, markersize=3,
                label='C Final', color='#17becf'
            )

        # styling
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)

        # y‑axis formatting & tick‐limit
        ax.yaxis.set_major_formatter(FuncFormatter(sec_to_time))
        ax.yaxis.set_major_locator(MaxNLocator(nbins=4, prune='both'))
        ax.tick_params(axis='y', labelsize=8)

        # x‑axis
        ax.set_xlim(min_year, max_year)
        ax.set_xticks(years_tick)
        ax.tick_params(axis='x', rotation=45, labelsize=8)

        # per‐subplot title
        ax.set_title(event_name.replace('_', ' '), fontsize=12, pad=5)

        # autoscale just y
        ax.autoscale(axis='y')

        # optional: show legend per plot
        # ax.legend(frameon=False, fontsize=8, loc='best')

    for ax in axes_flat:
        handles, labels = ax.get_legend_handles_labels()
        if handles:
            legend = fig.legend(
                handles, labels,
                loc='upper center',
                ncol=3,
                frameon=True,          
                fontsize=10,
                bbox_to_anchor=(0.5, 1.0),
                fancybox=True,      
                shadow=False
            )
            # style the legend box
            legend.get_frame().set_edgecolor('black')
            legend.get_frame().set_linewidth(1.0)
            legend.get_frame().set_facecolor('white')
            break


    for ax in axes_flat[len(event_data_dict):]:
        fig.delaxes(ax)

    fig.suptitle('Final Cutoff Times by Event and Year', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.subplots_adjust(top=0.95)
    out_path = Path(output_dir) / 'cutoffs_grid.png'
    fig.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

    return out_path

In [40]:
event_dict = {}
for event_id, group in df.groupby('event_id'):
    if group[['a_final_cutoff_sec','b_final_cutoff_sec','c_final_cutoff_sec']].notna().any(axis=None):
        event_dict[event_id] = group

grid_path = plot_cutoffs_grid(event_dict, output_dir)
print(f"Saved combined grid: {grid_path}")

Saved combined grid: ../output/plots/event_cutoffs/cutoffs_grid.png
